In [1]:
# training.py
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm

from preprocessing import CreditDataCleaner, WOEIVTransformer, VIFSelector
preprocess_pipeline = Pipeline([
    ("cleaner", CreditDataCleaner()),
    ("woe_iv", WOEIVTransformer(target="SeriousDlqin2yrs", iv_threshold=0.02, bin_num_limit=5)),
    ("vif", VIFSelector(threshold=5.0)),
])

Shape before drop duplicates: (150000, 11)
Shape after drop duplicates: (149391, 11)

Train target distribution:
SeriousDlqin2yrs
0    0.933003
1    0.066997
Name: proportion, dtype: float64

Valid target distribution:
SeriousDlqin2yrs
0    0.932996
1    0.067004
Name: proportion, dtype: float64


In [2]:
# Config
DATA_PATH = "data/cs-training.csv"
TARGET = "SeriousDlqin2yrs"
RND = 42

# 1) Load + split 80/20 stratified
df = pd.read_csv(DATA_PATH, index_col=0)
df = df.drop_duplicates()
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RND, stratify=y
)

# 2) Build pipeline (preprocess -> classifier) and GridSearchCV with 5-fold stratified CV
full_pipeline = Pipeline([
    ("preprocess", preprocess_pipeline),
    ("clf", LogisticRegression(max_iter=1000))
])

param_grid = {
    "clf__C": [0.01, 0.1, 1.0, 10.0],
    "clf__penalty": ["l1"],
    "clf__solver": ["liblinear"],  # liblinear supports l1/l2 for binary
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RND)
gs = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1
)

gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print("Best CV AUC:", gs.best_score_)

# Evaluate on validation set
best = gs.best_estimator_
y_pred_proba = best.predict_proba(X_valid)[:, 1]
valid_auc = roc_auc_score(y_valid, y_pred_proba)
print("Validation AUC:", valid_auc)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[INFO] creating woe binning ...

========== WOE / IV Selection ==========
Số biến ban đầu: 10
Số biến sau khi lọc IV >= 0.02: 9
Danh sách biến được giữ lại:
['NumberOfDependents', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'RevolvingUtilizationOfUnsecuredLines', 'MonthlyIncome', 'NumberOfTimes90DaysLate', 'NumberOfOpenCreditLinesAndLoans', 'NumberRealEstateLoansOrLines', 'DebtRatio']

IV table:
                               variable        iv
0  RevolvingUtilizationOfUnsecuredLines  1.083545
1               NumberOfTimes90DaysLate  0.788450
2  NumberOfTime30-59DaysPastDueNotWorse  0.683582
3                                   age  0.254551
4       NumberOfOpenCreditLinesAndLoans  0.081448
5                             DebtRatio  0.072363
6                         MonthlyIncome  0.071242
7          NumberRealEstateLoansOrLines  0.056521
8                    NumberOfDependents  0.035331
9  NumberOfTime60-89DaysPastDueNotWors

In [6]:
# ============================================================
# 3) Get processed features, drop MonthlyIncome_woe, fit statsmodels Logit
#    and evaluate AUC on validation set
# ============================================================

from sklearn.metrics import roc_auc_score
import statsmodels.api as sm
import numpy as np
import pandas as pd

preproc = best.named_steps["preprocess"]

# Transform train/valid bằng preprocessing đã fit từ best model
X_train_trans = preproc.transform(X_train)
X_valid_trans = preproc.transform(X_valid)

# Đảm bảo output là DataFrame có tên cột
if isinstance(X_train_trans, np.ndarray):
    X_train_trans = pd.DataFrame(
        X_train_trans,
        columns=[f"X{i}" for i in range(X_train_trans.shape[1])],
        index=X_train.index
    )
else:
    X_train_trans = X_train_trans.copy()
    X_train_trans.index = X_train.index

if isinstance(X_valid_trans, np.ndarray):
    X_valid_trans = pd.DataFrame(
        X_valid_trans,
        columns=X_train_trans.columns,
        index=X_valid.index
    )
else:
    X_valid_trans = X_valid_trans.copy()
    X_valid_trans.index = X_valid.index

# ============================================================
# Drop MonthlyIncome_woe
# ============================================================

drop_cols = ["MonthlyIncome_woe", "NumberOfDependents_woe"]

X_train_trans_drop = X_train_trans.drop(columns=drop_cols, errors="ignore")
X_valid_trans_drop = X_valid_trans.drop(columns=drop_cols, errors="ignore")

print("Original train transformed shape:", X_train_trans.shape)
print("After dropping MonthlyIncome_woe :", X_train_trans_drop.shape)

print("\nFinal features used in statsmodels:")
print(X_train_trans_drop.columns.tolist())

# ============================================================
# Fit statsmodels Logit
# ============================================================

X_sm = sm.add_constant(X_train_trans_drop, has_constant="add")
y_sm = y_train.loc[X_sm.index]

sm_model = sm.Logit(y_sm, X_sm)
sm_res = sm_model.fit(disp=False)

# ============================================================
# Predict valid and calculate AUC
# ============================================================

X_valid_sm = sm.add_constant(X_valid_trans_drop, has_constant="add")

# Đảm bảo cột valid đúng thứ tự như train
X_valid_sm = X_valid_sm[X_sm.columns]

y_valid_proba_sm = sm_res.predict(X_valid_sm)

valid_auc_sm = roc_auc_score(y_valid, y_valid_proba_sm)

print("\nValidation AUC after dropping features:")
print(valid_auc_sm)

# ============================================================
# Build summary table
# ============================================================

params = sm_res.params
bse = sm_res.bse
z_values = sm_res.tvalues
pvalues = sm_res.pvalues
conf = sm_res.conf_int()

or_values = np.exp(params)
conf_exp = np.exp(conf)

table = pd.DataFrame({
    "Predictor": params.index,
    "β": params.values,
    "SE": bse.values,
    "z value": z_values.values,
    "p-value": pvalues.values,
    "OR": or_values.values,
    "CI Lower": conf_exp.iloc[:, 0].values,
    "CI Upper": conf_exp.iloc[:, 1].values
})

table["OR (95% CI)"] = table.apply(
    lambda row: f"{row['OR']:.3f} ({row['CI Lower']:.3f}, {row['CI Upper']:.3f})",
    axis=1
)

table = table[
    [
        "Predictor",
        "β",
        "SE",
        "z value",
        "p-value",
        "OR (95% CI)"
    ]
]

pd.set_option("display.float_format", lambda x: f"{x:.6f}")

print("\nStatsmodels Logistic Regression Summary Table:")
print(table.to_string(index=False))

[INFO] converting into woe values ...
[INFO] converting into woe values ...
Original train transformed shape: (119512, 9)
After dropping MonthlyIncome_woe : (119512, 7)

Final features used in statsmodels:
['age_woe', 'NumberOfTime30-59DaysPastDueNotWorse_woe', 'RevolvingUtilizationOfUnsecuredLines_woe', 'NumberOfTimes90DaysLate_woe', 'NumberOfOpenCreditLinesAndLoans_woe', 'NumberRealEstateLoansOrLines_woe', 'DebtRatio_woe']

Validation AUC after dropping features:
0.8468981694490177

Statsmodels Logistic Regression Summary Table:
                               Predictor         β       SE     z value  p-value          OR (95% CI)
                                   const -2.622743 0.014128 -185.638401 0.000000 0.073 (0.071, 0.075)
                                 age_woe  0.475266 0.028885   16.453879 0.000000 1.608 (1.520, 1.702)
NumberOfTime30-59DaysPastDueNotWorse_woe  0.624716 0.014477   43.152355 0.000000 1.868 (1.815, 1.921)
RevolvingUtilizationOfUnsecuredLines_woe  0.649642 0.01

In [7]:
# ============================================================
# 4) Predict trên test set bằng statsmodels model đã drop features
#    và tạo submission.csv
# ============================================================

TEST_PATH = "data/cs-test.csv"

test_df = pd.read_csv(TEST_PATH, index_col=0)

# Với Give Me Some Credit, test có thể có cột SeriousDlqin2yrs toàn NaN
X_test = test_df.drop(columns=[TARGET], errors="ignore").copy()

print("Test raw shape:", X_test.shape)

# Transform test bằng preprocessing đã fit từ best model
X_test_trans = preproc.transform(X_test)

# Đảm bảo output là DataFrame có tên cột giống train transformed
if isinstance(X_test_trans, np.ndarray):
    X_test_trans = pd.DataFrame(
        X_test_trans,
        columns=X_train_trans.columns,
        index=X_test.index
    )
else:
    X_test_trans = X_test_trans.copy()
    X_test_trans.index = X_test.index

# Drop cùng các biến đã bỏ khi fit statsmodels
X_test_trans_drop = X_test_trans.drop(columns=drop_cols, errors="ignore")

print("Test transformed shape before drop:", X_test_trans.shape)
print("Test transformed shape after drop :", X_test_trans_drop.shape)

# Thêm constant
X_test_sm = sm.add_constant(X_test_trans_drop, has_constant="add")

# Đảm bảo đúng thứ tự cột như lúc fit statsmodels
X_test_sm = X_test_sm[X_sm.columns]

# Predict probability class 1
test_proba = sm_res.predict(X_test_sm)

print("Test probability summary:")
print(pd.Series(test_proba).describe())

# Tạo submission
submission = pd.DataFrame({
    "Id": np.arange(1, len(test_proba) + 1),
    "Probability": test_proba
})

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

print("\nSaved submission to:", submission_path)
print("Submission shape:", submission.shape)
print(submission.head())

Test raw shape: (101503, 10)
[INFO] converting into woe values ...
Test transformed shape before drop: (101503, 9)
Test transformed shape after drop : (101503, 7)
Test probability summary:
count   101503.000000
mean         0.067173
std          0.109866
min          0.007072
25%          0.014706
50%          0.025566
75%          0.069166
max          0.861360
dtype: float64

Saved submission to: submission.csv
Submission shape: (101503, 2)
   Id  Probability
1   1     0.116034
2   2     0.031385
3   3     0.016811
4   4     0.128849
5   5     0.116034
